Imports

In [11]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
import joblib
import os

os.makedirs('../outputs/models', exist_ok=True)
os.makedirs('../data/processed', exist_ok=True)

Load Raw Data

In [12]:
df = pd.read_csv('/home/ankit_samal/ML project/data/raw/Banglore_traffic_Dataset.csv')
print("Original shape:", df.shape)
df.head()

Original shape: (8936, 16)


,Date,Area Name,Road/Intersection Name,Traffic Volume,Average Speed,Travel Time Index,Congestion Level,Road Capacity Utilization,Incident Reports,Environmental Impact,Public Transport Usage,Traffic Signal Compliance,Parking Usage,Pedestrian and Cyclist Count,Weather Conditions,Roadwork and Construction Activity
0,2022-01-01,Indiranagar,100 Feet Road,50590,50.230299,1.500000,100.000000,100.000000,0,151.180,70.632330,84.044600,85.403629,111,Clear,No
1,2022-01-01,Indiranagar,CMH Road,30825,29.377125,1.500000,100.000000,100.000000,1,111.650,41.924899,91.407038,59.983689,100,Clear,No
2,2022-01-01,Whitefield,Marathahalli Bridge,7399,54.474398,1.039069,28.347994,36.396525,0,64.798,44.662384,61.375541,95.466020,189,Clear,No
3,2022-01-01,Koramangala,Sony World Junction,60874,43.817610,1.500000,100.000000,100.000000,1,171.748,32.773123,75.547092,63.567452,111,Clear,No
4,2022-01-01,Koramangala,Sarjapur Road,57292,41.116763,1.500000,100.000000,100.000000,3,164.584,35.092601,64.634762,93.155171,104,Clear,No


Parse Date & Extract Features

In [13]:
df['Date'] = pd.to_datetime(df['Date'])

df['month']       = df['Date'].dt.month
df['day_of_week'] = df['Date'].dt.dayofweek
df['day']         = df['Date'].dt.day
df['is_weekend']  = df['day_of_week'].isin([5, 6]).astype(int)
df['quarter']     = df['Date'].dt.quarter

print("New features added: month, day_of_week, day, is_weekend, quarter")
df[['Date', 'month', 'day_of_week', 'is_weekend', 'quarter']].head()

New features added: month, day_of_week, day, is_weekend, quarter


,Date,month,day_of_week,is_weekend,quarter
0,2022-01-01,1,5,1,1
1,2022-01-01,1,5,1,1
2,2022-01-01,1,5,1,1
3,2022-01-01,1,5,1,1
4,2022-01-01,1,5,1,1


Drop Leakage & Irrelevant Columns

In [14]:
# Environmental Impact = 1.0 correlation = derived from target = data leakage
# Date = already extracted, no longer needed
# Road/Intersection Name = too granular, Area Name is enough

drop_cols = [
    'Date',
    'Road/Intersection Name',
    'Environmental Impact',
    'Travel Time Index',        # leakage — effect of traffic volume
    'Congestion Level',         # leakage — effect of traffic volume
    'Road Capacity Utilization' # leakage — effect of traffic volume
]
df.drop(columns=drop_cols, inplace=True)


print("Dropped:", drop_cols)
print("Remaining columns:", df.columns.tolist())

Dropped: ['Date', 'Road/Intersection Name', 'Environmental Impact', 'Travel Time Index', 'Congestion Level', 'Road Capacity Utilization']
Remaining columns: ['Area Name', 'Traffic Volume', 'Average Speed', 'Incident Reports', 'Public Transport Usage', 'Traffic Signal Compliance', 'Parking Usage', 'Pedestrian and Cyclist Count', 'Weather Conditions', 'Roadwork and Construction Activity', 'month', 'day_of_week', 'day', 'is_weekend', 'quarter']


Encode Categorical Columns

In [15]:
le_area    = LabelEncoder()
le_weather = LabelEncoder()
le_road    = LabelEncoder()

df['Area Name']                      = le_area.fit_transform(df['Area Name'])
df['Weather Conditions']             = le_weather.fit_transform(df['Weather Conditions'])
df['Roadwork and Construction Activity'] = le_road.fit_transform(df['Roadwork and Construction Activity'])

# Save encoders for Flask backend later
joblib.dump(le_area,    '../outputs/models/le_area.pkl')
joblib.dump(le_weather, '../outputs/models/le_weather.pkl')
joblib.dump(le_road,    '../outputs/models/le_road.pkl')

print("Encoding done. Saved encoders.")
df.head()

Encoding done. Saved encoders.


,Area Name,Traffic Volume,Average Speed,Incident Reports,Public Transport Usage,Traffic Signal Compliance,Parking Usage,Pedestrian and Cyclist Count,Weather Conditions,Roadwork and Construction Activity,month,day_of_week,day,is_weekend,quarter
0,2,50590,50.230299,0,70.632330,84.044600,85.403629,111,0,0,1,5,1,1,1
1,2,30825,29.377125,1,41.924899,91.407038,59.983689,100,0,0,1,5,1,1,1
2,6,7399,54.474398,0,44.662384,61.375541,95.466020,189,0,0,1,5,1,1,1
3,4,60874,43.817610,1,32.773123,75.547092,63.567452,111,0,0,1,5,1,1,1
4,4,57292,41.116763,3,35.092601,64.634762,93.155171,104,0,0,1,5,1,1,1


Define Features & Target

In [16]:
TARGET = 'Traffic Volume'

FEATURES = [col for col in df.columns if col != TARGET]

X = df[FEATURES]
y = df[TARGET]

print("Features:", FEATURES)
print("Target  :", TARGET)
print("X shape :", X.shape)
print("y shape :", y.shape)

Features: ['Area Name', 'Average Speed', 'Incident Reports', 'Public Transport Usage', 'Traffic Signal Compliance', 'Parking Usage', 'Pedestrian and Cyclist Count', 'Weather Conditions', 'Roadwork and Construction Activity', 'month', 'day_of_week', 'day', 'is_weekend', 'quarter']
Target  : Traffic Volume
X shape : (8936, 14)
y shape : (8936,)


Train/Test Split


In [17]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training samples  :", X_train.shape[0])
print("Testing samples   :", X_test.shape[0])

Training samples  : 7148
Testing samples   : 1788


Scale Features

In [18]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)  # only transform, never fit on test

# Save scaler for Flask backend
joblib.dump(scaler, '../outputs/models/scaler.pkl')

print("Scaling done. Scaler saved.")
print("X_train_scaled shape:", X_train_scaled.shape)

Scaling done. Scaler saved.
X_train_scaled shape: (7148, 14)


Save Processed Data


In [19]:
# Save for use in modeling notebook
np.save('../data/processed/X_train.npy', X_train_scaled)
np.save('../data/processed/X_test.npy',  X_test_scaled)
np.save('../data/processed/y_train.npy', y_train.values)
np.save('../data/processed/y_test.npy',  y_test.values)

# Save feature names for Flask
joblib.dump(FEATURES, '../outputs/models/feature_names.pkl')

print("All processed data saved to data/processed/")
print("Features list saved to outputs/models/")

All processed data saved to data/processed/
Features list saved to outputs/models/


Verify

In [20]:
print("=== Preprocessing Complete ===")
print(f"Total features used : {len(FEATURES)}")
print(f"Training samples    : {X_train_scaled.shape[0]}")
print(f"Test samples        : {X_test_scaled.shape[0]}")
print(f"\nFeature list:")
for i, f in enumerate(FEATURES, 1):
    print(f"  {i:2}. {f}")

=== Preprocessing Complete ===
Total features used : 14
Training samples    : 7148
Test samples        : 1788

Feature list:
   1. Area Name
   2. Average Speed
   3. Incident Reports
   4. Public Transport Usage
   5. Traffic Signal Compliance
   6. Parking Usage
   7. Pedestrian and Cyclist Count
   8. Weather Conditions
   9. Roadwork and Construction Activity
  10. month
  11. day_of_week
  12. day
  13. is_weekend
  14. quarter
